我目前的训练数据包括粗粒度图结构数据和对应的核磁数据，这些数据取自有机小分子，最大碳数量不超过80。粗粒度分子是对全原子分子的粗粒度转化，具体包括33个粗粒度结构单元su，其中26个是含碳结构单元，且这些结构单元只含有一个碳原子，其余7个为不含碳结构单元，每个结构单元除H外其余原子均是固定的，部分结构单元的H原子是变化的（根据其连接的周围结构单元数量决定）；粗粒度的图结构数据包括节点特征矩阵x（总计39维数据，0-32为one-hot su编码，33-38为结构单元的元素组成CHONSX（X为卤组元素）），邻接矩阵edge_index和边特征矩阵edge_attr（2维数据，0-1,分别表示键级，是否在环上两个特征），全局元素组成total_atom_counts（CHONSX 6维数据），全局结构单元分布su_hist（33维结构单元one-hot）；核磁数据是0-300ppm，总计3001个数据点的向量数据，使用了洛伦兹函数对核磁的峰进行扩展，生成固定半峰宽为1ppm的洛伦兹峰，注意核磁数据为稀疏数据，大部分位置没有峰，强度为0，核磁数据经过峰强度归一化，最大峰强度为1；此外，还有含碳节点对应的节点级核磁信息，包括每个含碳节点对应的峰中心和强度。

首先应该直接根据得到S2N预测得到结构单元分布H_target直接对每个su分配1-hop类型（所有的su，包括非碳su，全部分配1-hop，这些1-hop应该来自子图数据库中每类结构单元su存在的1-hop组成类型），然后查看这些模版的1-hop是否满足H_target的约束，每个中心su（假设su是甲烷CH3-，只能外接一个结构单元su，那么1-hop就需要满足所有的中心su的su_max_degree（1）x H_target（CH3-））个该类su），所有的su的1-hop组成分布结果需要硬性满足H_target。

然后，为已经确定好的中心su+1-hop从子图数据库选取存在2-hop类型，及针对每个su分配完整的子图模版，注意这里的2-hop同样会受到H_target的约束（假设su是-CH2-，能外接两个结构单元su，假设这两个su分别是-CH2-和-O-，那么该中心su-CH2-就能够在连接两个2-hop节点，对应的2-hop只允许存在两个该类su，就是说2-hop满足的约束为每个su连接的1-hop的su类型的su_max_degree之和减去1-hop节点数量），并分配每个模版对应的z（每个模版对应的核磁峰居中的那个特征z，由子图数据库中进行过统计的），然后计算含碳节点的核磁，并与目标核磁进行比较，然后调整模版,就是调整2-hop，使其能够向目标核磁谱图靠近。

最后再细选每个模版库存在的z，根据已经重构出来的核磁与目标核磁在强度缺口处的差异，选取更加合适的z.  这样分为多层来实现模版的选取和最终z的选取。这里强调1-hop的强约束满足H_target，2-hop可以适当放宽H_target的约束。主要的难点是怎么调整H_target使得最终的H_target以及子图模版，z及其最终预测的核磁结果能够高度匹配目标核磁。


1-hop和2-hop需要满足H_target约束的意思是：假如存在节点A（度数为2）：1个，B节点（度数为2）：2个，C节点（度数为3）：1个，组成子图B-A-B-C；对于1-hop，由于A的度数为2，那么单个A节点能够出现在1-hop两次，当以B节点为中心节点时，子图中1-hop可以出现两个A节点（A的中心节点只有1个，但是由于A的度数为2，因此A可以在1-hop出现2次）；对于2-hop，由于A连接的1-hop节点是两个B节点，且B节点的度数为2，因此A在2-hop中出现的次数就等于1-hop节点的度数之和减去1-hop的节点总数，及两个B节点（度数为2）能否连接4个其余节点，减去1-hop节点本身与中心节点的连接数2，因此子图中2-hop能够出现A节点的次数也是2个，及中心节点在其余子图中以2-hop节点出现的次数为该中心节点的1-hop节点的度数之和减去该中心节点的度数。

### layer0的修正

暂时不急于完成如reconstruct_spectrum！先完成Layer0！首先在输入目标核磁谱图和元素组成之后，使用S2N模型预测对应的SU组成与分布（必须是整数）！获得初始化的结果！然后针对这个初始化的结果进行优化调整！首先优化元素杂原子XSNO的结构单元数量！所有结果单元的元素固定为这个！首先需要调整初始的结构单元分布。例如，初始分布中存在大量的X元素（假设32号存在8个），但是实际的X元素含量只有3个，那么就存在多余的5个32号，需要删除，反之则补充！

同理，对于S元素（只存在30和31号为S），30和31的总数应该等于输入的元素总量的S，多余删除，优先保留31号，若不足则补充。删除/补充逻辑如下：如果当前已有的30号和31号中（数量分别是m，n），与目标S差多个S，补充/删除30和31号的个数使其等于目标S，补充/删除的方向是使的m、n的比例向0.4：0.6接近。注意不同时进行补充和删除，少于目标S时，就只做补充，多余目标，就只做删除。注意满足数量作为第一优先级，比例作为第二优先级，满足数量这第一优先级后，就无需满足比例。例如，当前有m=2，n=1，目标S=5，那么需要补充31号2个，这样m=2，n=3；当前有m=1，n=2，目标S=5，那么需要补充30和31号各1个，这样m=2，n=3；当前有m=3，n=0，目标S=5，那么需要补充31号2个，这样m=3，n=2（原本存在多少个31或者30号不变，增加的数量靠近比例0.4：0.6）；当前有m=0，n=3，目标S=5，那么需要补充30号2个，这样m=2，n=3（靠近比例0.4：0.6）；当前有m=1，n=2，目标S=5，那么需要补充30和31号各1个，这样m=2，n=3。对于结构单元的删减也是这样的，如果30和31号过多，例如m=5，n=2，目标S只有5个，那么需要删除30号2个，这样m=3，n=2（靠近比例0.4：0.6）；如果m=2，n=5，那么需要删除31号2个，这样m=2，n=3。（整体的逻辑与后续的N是一样的）

对于氮元素，只存在0号，4号，26和27号，这四个总数应该等于目标N的总数。其中优先级26>27>0>4，超出目标N，优先删除4，其次是0，然后是27和26；0,4,26,27的比例为0.1:0.05:0.45:0.4。设当前已有的0号x个，4号y个，26号z个，27号w个，如果x+y+z+w>目标N或者<目标N，那么需要进行删除/补充，使得x+y+z+w=目标N，删除/补充的逻辑是使得x:y:z:w的比例接近0.1:0.05:0.45:0.4，不同时进行补充和删除，少于目标时，就只做补充，多余目标，就只做删除！假设当前有0号3个，4号2个，26号3个，27号2个，目标N=10，x+y+z+w=10，不进行删除/补充，虽然不满足比例，但是满足数量作为第一优先级。假设当前有0号3个，4号2个，26号2个，27号1个，目标N=10，x+y+z+w=8<10，补充1个26号和1个27号，这样x+y+z+w=10，比例接近0.1:0.05:0.45:0.4。假设当前有0号3个，4号2个，26号4个，27号3个，目标N=10，x+y+z+w=12>10，删除1个0号和1个4号，这样x+y+z+w=10，比例接近0.1:0.05:0.45:0.4（需要根据当前比例和目标比例计算出需要补充/删除结构单元的个数）。

如何基于当前和目标比例差异来计算需要补充/删除结构单元的个数？对于N，逻辑如下：假设目前的0、4、26、27和数量为m、n、p、q，目标N为W，目标个数分布为0.1W、0.05W、0.45W、0.4W。计算（m-0.1W），(n-0.05W),(p-0.45W),(q-0.4W)的差（假设差值为A、B、C、D），如果m+n+p+q<W，及需要进行N补充，补充A、B、C、D中为负值的个数，依次补充负值最多的那个（先确定负值最多的那个，补充对应的1个结构，然后继续确定最多的负值，及每次补充1个最多的那个负值对应的结构单元），直至m+n+p+q=W；如果m+n+p+q>W，及需要进行N删除，删除A、B、C、D中为正值的个数，依次删除正值最多的那个（先确定正值最大的那个，删除对应的1个结构，然后继续确定最大的正值，及每次删除1个最大的那个正值对应的结构单元），直至m+n+p+q=W。例如，假设目标N是10个，那么目标0、4、26、27的个数分布为1,0.5,4.5和4，假设当前的0号m=3，4号n=2，26号p=2，27号q=1，x+y+z+w=8<10需要补充2个N，差值A=3-1=2，B=2-0.5=1.5，C=2-4.5=-2.5，D=1-4=-3，其中C和D为负值，其中D负值最多，先补充一个27号及q=2这样D变为-2，然后C就变为负值最多，再补充一个26号，这样q=3，C变为-1。补充完2个N后，m+n+p+q=10，不再补充。假设当前的0号m=3，4号n=2，26号p=4，27号q=3，x+y+z+w=12<10需要删除2个N，差值A=3-1=2，B=2-0.5=1.5，C=4-4.5=-0.5，D=3-4=-1，其中A、B为正，其中A最大正值，先删除一个0号及m=2使得A=1，删除之后B的正值就成最大的，因此再删除一个4号及n=1使得B为0.5，删除完2个N后，m+n+p+q=10，不再删除。

对于O，包括了0、1（2个O）、2（2和O）、3号，28,29号（除1,2外其余的均只有1个O），这些结构单元的O总数应该与输入的O总数一致。其中结构单元的优先级为29>28>1>3>2>0，比例为0.025:0.25:0.1：0.2：0.2：0.225，删除/补充逻辑如下：如果当前已有的0号、1号、2号、3号、28号、29号中（数量分别是m，n，p，q，r，s），与目标O差（m+2n+2p+q+r+s-目标O的差异），补充/删除0号、1号、2号、3号、28号、29号的个数使其等于目标O，补充/删除的方向是使的m、n、p、q、r、s的比例向0.025:0.25:0.1：0.2：0.2：0.225接近。注意不同时进行补充和删除，少于目标O时，就只做补充，多余目标，就只做删除！假设当前有0号3个，1号2个，2号2个，3号1个，28号3个，29号2个，目标O=17，m+2n+2p+q+r+s=17，不进行删除/补充，虽然不满足比例，但是满足数量作为第一优先级。假设当前有0号2个，1号3个，2号0个，3号1个，28号3个，29号4个，目标O=20，m+2n+2p+q+r+s=16<20，补充1个2号和2个28号，这样m+2n+2p+q+r+s=20，比例接近0.025:0.25:0.1：0.2：0.2：0.225。假设当前有0号2个，1号3个，2号3个，3号4个，28号4个，29号3个，目标O=20，m+2n+2p+q+r+s=25>20，删除1个0号和1个1号以及2个29号，这样m+2n+2p+q+r+s=20，比例接近0.025:0.25:0.1：0.2：0.2：0.225（需要根据当前比例和目标比例计算出需要补充/删除结构单元的个数）。

如何基于当前和目标比例差异来计算需要补充/删除结构单元的个数？对于O，逻辑与N相同，如下：假设目前的0、1、2、3、28、29和数量为m、n、p、q、r、s，目标N为W，目标O的个数分布为0.025W、0.25W、0.1W、0.2W、0.2W、0.225W。计算（m-0.025W），(2n-0.25W),(2p-0.1W),(q-0.2W),(r-0.2W),(s-0.225W)的差（假设对应差值为A、B、C、D、E、F），如果m+2n+2p+q+r+s<W，及需要进行O补充，补充A、B、C、D、E、F中为负值的个数，依次补充负值最多的那个（先确定负值最多的那个，补充对应的1个结构，然后继续确定最多的负值，及每次补充1个最多的那个负值对应的结构单元），直至m+2n+2p+q+r+s=W；如果m+2n+2p+q+r+s>W，及需要进行N删除，删除A、B、C、D、E、F中为正值的个数，依次删除正值最多的那个（先确定正值最大的那个，删除对应的1个结构，然后继续确定最大的正值，及每次删除1个最大的那个正值对应的结构单元），直至m+2n+2p+q+r+s=W。例如，假设目标O是20个，那么目标0、1、2、3、28、29的O个数分布为0.5，5,2，4，4，4.5，假设当前的0号m=2，1号n=3，2号p=0，3号q=1，28号r=3，29号s=4，m+2n+2p+q+r+s=16<20需要补充4个O，差值A=2-0.5=1.5，B=3*2-5=1，C=0-2=-2，D=1-4=-3，E=3-4=-1，F=4-4.5=-0.5，其中其中D负值最多，先补充一个3号及q=2这样D变为-2，然后C和D就变为负值最多且一样（3号的优先级高，先补3号），再补充一个3号，这样q=3，D变为-1；然后C负值最多，补一个2号（2个O），p=1，此时m+2n+2p+q+r+s>=20（注意是>=,存在最后补充1,2号情况，但是之差一个O，补充之后就多一个O，使得m+2n+2p+q+r+s>目标O，也视为结束）,不在补。假设当前的0号m=2，1号n=3，2号p=3，3号q=4，28号r=4，29号s=3，m+2n+2p+q+r+s=25>20需要删除5个O，差值A=2-0.5=1.5，B=3*2-5=1，C=3*2-2=4，D=4-4=0，E=4-4=0，F=3-4.5=-1.5，其中C最大正值，先删除一个3号及p=2使得C=2，删除之后C的正值还是最大的，因此再删除一个3号及p=1使得C=0，此时0号的A值最大，删除1个0号，使得m=2及A=0.5，此时m+2n+2p+q+r+s<=20（注意是<=,存在最后删除1,2号情况，但是之差一个O，删除之后就少一个O，使得m+2n+2p+q+r+s<目标O，也视为结束）,不在删除！

接下来需要修正含碳结构单元的组成！前面我们已经修正了X, S, N, O的元素组成，现在需要修正C, H的元素组成。结构单元之间是存在固定的连接匹配关系。

O元素对应了多种形式，0号（-C=ONH-，双端桥接结构，连接两个结构单元）、1号（COOH）、2号（-COO-，双端桥接结构，连接两个结构单元）、3号（-C=O-，双端桥接结构，连接两个结构单元）、28（-OH）、29（-O-，双端桥接结构，连接两个结构单元）6种类型。其中0、1、2、3这四种中的C=O存在结构单元9号（芳香卤取代碳Car）的固定连接，同时也可以连接如22、23、24等结构，连接形式较多，要求必须存在一定量的9号用于C=O结构的连接。其中先计算出C=O结构总连接量=0号*1+1号*1+2号*1+3号*2=W，及需要的9号的数量为W*0.6，取整。其中2、28、29中存在-O端，存在固定连接5号（芳香醚取代碳Car）和19号（醚接脂肪碳-CH2-）。假设需要的5号m个、19号n个，总量为m+n=2号*1+28号*1+29号*2=W,其中优先级为19>5，比例为0.55:0.45。若m+n<W，及不相等需要进行补充，计算（m-0.55W），(n-0.45W)的差（假设差值为A、B），补充A、B中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的19号），直至m+n=W；如果m+n>W，及需要进行删除，删除A、B中为正值的个数，依次删除正值最多的那个（若负值一样，则先删除优先级低的5号），直至m+n=W。

然后是S元素对应了30号（芳香环中的杂原子S）和31号（硫醚-S-，双端桥接结构，连接两个结构单元）两种形式，其中30号可以连接所有芳香结构（5-13号），不做固定约束；31号与结构单元（7号（芳香硫醚取代碳Car）、19号（硫醚/醚接脂肪碳-CH2-））连接，优先级19>7，比例为0.6:0.4，需要7号m个和19号n个的数量和m+n=31号*2=W，多则删除，少则补充。注意，由于O和S共用了19号，前面已经修正的19号的数量只用于-O-，因此对于31号需要的19号需要单独计算，视为n=0，后续修正得到的n增加到原本O修正之后的19号上。同理，若m+n（n=0）<W，及不相等需要进行补充，计算（m-0.55W），(n-0.45W)的差（假设差值为A、B），补充A、B中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的19号），直至m+n=W；如果m+n（n=0）>W，及需要进行删除，删除A、B中为正值的个数，依次删除正值最多的那个（若负值一样，则先删除优先级低的7号），直至m+n=W。

其次对于N，0号（-C=ONH-）、27号（-NH-）这种形式的氨基N必然与结构单元（6号（芳香氨基取代碳Car）、20号（氨基接脂肪碳-CH2-））连接，至于4号-CN没有固定匹配，26号N是芳香环中的杂原子N，也不计算固定配对，因此只需要计算0号（-C=ONH-）、27号（-NH-）所需的固定配对6号和20号的数量。优先级6>20，比例为0.6:0.4。假设0号x个、27号y个，那么所需8号m个和21号n个的数量和就应该为x+2y=m+n，多则删除，少则补充。逻辑同先前的N元素修正，逻辑如下：假设目前x+2y=W，目标6号和20号分布为0.6W、0.4W。计算（m-0.6W），(n-0.4W)的差（假设差值为A、B），如果m+n<W，及需要进行补充，补充A、B中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的6号），直至m+n=W；如果m+n>W，及需要进行删除，删除A、B中为正值的个数，依次删除正值最多的那个（若负值一样，则先删除优先级低的20号），直至m+n=W。

X（32号，末端结构，只连接一个结构单元）与结构单元（8号（芳香卤取代碳Car）、21号（卤接脂肪碳-CH2-））的连接匹配关系是固定的，也就是说一个32号必然对应一个8号或一个21号。优先级8>21，比例为0.6:0.4，假设32号存在4个，那么8号和21号的数量和就应该为4，多则删除，少则补充。逻辑同先前的N元素修正，辑如下：假设目前32号-X存在W个，8号和21号的数量为m、n，目标8号和21号分布为0.6W、0.4W。计算（m-0.6W），(n-0.4W)的差（假设差值为A、B），如果m+n<W，及需要进行补充，补充A、B中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的8号），直至m+n=W；如果m+n>W，及需要进行删除，删除A、B中为正值的个数，依次删除正值最多的那个（若负值一样，则先删除优先级低的21号），直至m+n=W。

存在问题，修正的顺序需要调整一下！首先依旧修正XSN三种元素，然后需要修正的是C=O的数量分布，这个修正先于O的修正！修正羰基时，需要计算输入的光谱0-90脂肪区域（包括结构单元id 19、20、21、22、23、24、25）和90-160芳香区域（包括结构单元id 4、5、6、7、6、9、10、11、12、13、14、15、16、17、18）和160-240羰基区域（包括结构单元id 0、1、2、3）的比例，面积比，假设为x，y，z。然后分配元素C的数量，假设总C为N个，则三个光谱区域的C数量分别为0.82*xN，N-0.82*xN-1.35*zN，1.35*zN。

a、其中羰基结构数量zN，表示C=O结构的数量。所以需要充分分配0,1,2,3的数量。由于N的修正已经分配了0号结构！所以后续保持0号不变。只修正结构单元1,2,3.分配比例为，0.35，0.25,0.4，优先级3>1>2.假设目前的1、2、3和数量为m、n、p，设定目标需要修正的羰基结构为W=zN-0号个数，目标C=O的个数分布为0.35W、0.25W、0.4W。计算m+n+p是否等于W，多则删除，少则补充。修正逻辑如下：计算（m-0.35W），(n-0.25W)，（p-0.4W）的差（假设差值为A、B、C），若m+n+p<W，补充其中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的），直至m+n+p+q=W；如果m+n+p>W，及需要进行删除，删除其中为正值的个数，依次删除正值最多的那个（若正值一样，则先删除优先级低的），直至m+n+p=W。

存在特殊情况，关于羰基的修正需要注意的是羰基的O总数，按照比例计算得到的羰基的总数W=zN，其中zN的O总数可能会超过了目标O的总数！假设目标O的数量为M，0,1,2,3的数量分别为m、n、p、q，当前的O总数为M1=m+n*2+p*2+q。当M1>M时候，表示当前的羰基过多，O过多，超过目标O的上限！此时优先满足O总数等于目标氧总数，不能满足羰基总数等于按照谱图比例计算得到的羰基的总数！修正逻辑如下：依旧保存0号结构不变，只修正结构单元1,2,3，其O的分配比例为0.35，0.25,0.4，优先级1>2>3.假设目前的1、2、3和数量为m、n、p，设定目标需要修正的O数量为Y=目标O总数M-0号个数，目标O的个数分布为0.35W、0.25W、0.4W。计算当前的2*m+2*n+p与Y的差，必然是多于Y，需要减少结构单元。修正逻辑如下：计算（2*m-0.35W），(2*n-0.25W），（p-0.4W）的差（假设差值为A、B、C），依次删除正值最多的那个（若正值一样，则先删除优先级低的），直至2*m+2*n+p=Y。然后，计算修正后的C=O的总数=m+n+p+0号的数量=Z，这个Z必然是小于目标W的，计算W-Z及与目标羰基结构数量的差值，这部分W-Z的数量补充到芳香结构总数中以满足碳总数守恒！及当前的芳香碳总数为yN+0.1*xN+（W-Z）。及90-160ppm对应的碳总数为yN+0.1*xN+（W-Z），不再是yN+0.1*xN！此外，由于此时O已经被占满，后续不在需要进行O的修正！及不需要修正O结构单元28,29号，且28,29号的数量必须归0，因为O已经被占满！

b、然后再进行O的修正，N修正和羰基修正了0,1,2,3这4个含氧结构单元，剩下的含氧结构单元只剩下28,29.假设目标O的数量是M，需要修正的O数量是M-0号-1号*2-2号*2-3号=W，假设28号m个、29号n个，28,29的比例为0.55:0.45，优先级28>29,目标分布为0.55W，0.45W。计算当前的m+n是否等于W，多则删除，少则补充。修正逻辑如下：若m+n<W，及不相等需要进行补充，计算（m-0.55W），(n-0.45W)的差（假设差值为A、B），补充A、B中为负值的个数，依次补充负值最多的那个（若负值一样，则先补充优先级高的28号），直至m+n=W；如果m+n（n=0）>W，及需要进行删除，删除A、B中为正值的个数，依次删除正值最多的那个（若负值一样，则先删除优先级低的29号），直至m+n=W。

这样就完成了X、S、N、C=O、O的修正。然后修正结构单元之间存在固定的连接匹配关系，保持先前的固定匹配结构单元的数量修正逻辑和顺序，及修正C=O对应的9号，O对应的5号+19号，S对应的7号+19号，N对应的6号+20号，X对应的8号+21号。这样就修正了0、1、2、3、4、5、6、7、8、9、19、20、21号，以及26、27、28、29、30、31、32号。还剩下10,11,12,13芳香结构，非饱和碳14,15,16,17,18，以及脂肪碳22,23,24,25，及需要修正C、H含量。

前面得到的（假设总C为N个，则三个光谱区域0-90、90-160、160-240的C数量分别为xN，yN，zN）。计算当前需要修正的脂肪结构含量M=xN-19号-20号-21号，需要修正的芳香结构数量W=yN-4号-5号-6号-7号-8号-9号。

首先修正脂肪碳结构，假设脂肪碳含量0.82*xN=M，当前的22、23、24、25分别有m、n、p、q，直接修正m=0.2M,n=0.68M-19号-20号-21号,p=0.1M,q=0.02M。

对于非饱和结构，设数量为W=0.05*（N-0.82*xN-1.35*zN），由于非饱和结构成对出现，因此w取最近偶数。其中14、15、16、17、18分别有m、n、p、q、r。m+n+p：q+r=0.8:0.2（双键：三键），m+n+p=0.8W，取最近偶数；q+r=0.2W，取最近偶数。其中m：n：p=0.1:0.65:0.25，取整数使得m+n+p=0.8w.其中q：r=0.5:0.5均分0.2W。 

对于芳香结构数量为W=0.95*（N-0.82*xN-1.35*zN）-4号。其中假设10、11、12、13的数量为m、n、p、q,12号的数量需要基于芳香碳（包括非饱和碳）的数量比例来确定,计算（yN+0.1*xN）/C总数=fa，
如果fa<=0.5,则12号p=0.15W，10号m=0.1W，13号q=0.53W,11号n=0.22W-5号-6号-7号-8号-9号；
如果0.6>=fa>0.5,则12号p=0.18W，10号m=0.1W，13号q=0.50W,11号n=0.22W-5号-6号-7号-8号-9号；
如果0.7>=fa>0.6,则12号p=0.2W，10号m=0.1W，13号q=0.5W,11号n=0.20W-5号-6号-7号-8号-9号;
如果0.75>=fa>0.7,则12号p=0.225W，10号m=0.09W，13号q=0.485W,11号n=0.20W-5号-6号-7号-8号-9号;
如果0.8>=fa>0.75,则12号p=0.255W，10号m=0.085W，13号q=0.47W,11号n=0.19W-5号-6号-7号-8号-9号;
如果0.85>=fa>0.8,则12号p=0.285W，10号m=0.08W，13号q=0.46W,11号n=0.175W-5号-6号-7号-8号-9号;
如果0.9>=fa>0.85,则12号p=0.305W，10号m=0.07W，13号q=0.445W,11号n=0.17W-5号-6号-7号-8号-9号;
如果fa>0.9,则12号p=0.34W，10号m=0.07W，13号q=0.425W,11号n=0.165W-5号-6号-7号-8号-9号;

然后是H的调整！计算目前的H数量，与目标H数量的差值为ΔH。根据ΔH的正负来调整H的数量。ΔH/目标H<3%,不做调整，超过则需要增减来调整。分三个区域进行H的调整：设当前的总H为M，目标H为N，需要调整的H量为W=|M-1.02*N|，及调整到ΔH/目标H<2%。其中芳香区域调整的H量X=W*0.4；脂肪碳区域调整的H量Y=W*0.3；非饱和碳区域调整的H量Z=W*0.3。

- 如果ΔH为正，需要减少H。调整逻辑为：假设芳香区域需要减少X个H，调整中每次减少一个13号，轮流增加一个12号/12号/10号/11号，循环直到减少的H数量大于等于X个；假设脂肪区域需要减少Y个H，调整中第一轮每次减少一个22号，轮流增加一个23号/23号/23号/24号；第二轮，每次减少一个23号，增加24号；然后再循环到第一轮，第二轮，直到减少的H数量大于等于Y个；假设非饱和碳区域需要减少Z个H，调整中第一轮每次减少一个16号，轮流增加一个15号/14号；第二轮每次减少一个15号，增加一个14号；然后循环第一轮和第二轮，直到需要减少的H数量大于等于Z个。

- 如果ΔH为负，需要增加H。调整逻辑为：假设芳香区域需要增加X个H，调整中每次增加一个13号，轮流减少一个11号/12号/10号，循环直到增加的H数量大于等于X个；假设脂肪区域需要增加Y个H，调整中第一轮每次增加一个22号，轮流减少一个23号/23号/23号/24号/25号；第二轮，每次增加一个23号，减少24号；然后再循环到第一轮，第二轮，直到增加的H数量大于等于Y个；假设非饱和碳区域需要增加Z个H，调整中第一轮每次增加一个16号，轮流减少一个15号/14号；第二轮每次增加一个15号，减少一个14号；然后循环第一轮和第二轮，直到需要增加的H数量大于等于Z个。注意：调整H时，要保证每个区域的SU数量不能为负数。且在增加16号时，16的数量不能超过15号+14号的数量和。

限制25号的数量不超过3%的脂肪碳总量！22号的数量不少于23号的10%。

### layer1:1-hop的分配

结构单元之间的连接定义如下：所有结构单元（0-32号结构单元）的存在形式以及连接准则：
### A 类（0-2, 3-4）羰/腈（外接与必配）
- 0 `Amide_Group`（连接度=2）
  - 形式: -C(=O)-NH- 双端桥接单元，形式固定，不考虑-C(=O)-NH2/-C(=O)-N<形式。
  - 可连: C=O 端可连，按优先级 `9/22/23/24/25/14/15/17`；N 端固定可连 `6/20`；。
  - 禁止: 该结构单元不做末端。
- 1 `Carboxylic_Acid`（连接度=1）
  - 形式: 末端 -COOH。
  - 可连: 按优先级`9/23/24/25/14/15/17`。
  - 禁止: 不通过 `29` 转酯；不与其余末端结构相连（例如22：-CH3，28：-OH，16: Vinyllic_CH2，18: Alkynyl_CH）。
- 2 `Ester_Group`（连接度=2）
  - 形式: -C(=O)-O-，双端桥接。
  - 可连: 羰端可连，按优先级`9/23/24/25/14/15/17`；醇端固定连接 `5/19`。
  - 禁止: 该结构单元不可转为末端结构。
- 3 `Aldehyde_Ketone_C`（连接度=2）
  - 形式: -C(=O)-，可桥接。
  - 可连: 可连2个结构，按优先级`9/22/23/24/25/14/15/17`。
  - 禁止: 作为-C(=O)-时，不能两端同时连接末端结构单元。
- 4 `Nitrile_C`（连接度=1）
  - 形式: 末端 -C≡N。
  - 可连: 按优先级`23/24/25/10`（末端封端）。
  - 禁止: 不可连接区域末端结构。

### B 类 芳香（5-13）（固定成芳香环结构）
- 12 `Aromatic_Bridgehead_C`（连接度=3）
  - 形式: 稠环桥头碳。
  - 可连: 连接3个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香稠环）,固定包含至少一个12号结构单元。
  - 规则: 仅用于团簇内部成环，禁止外接，且邻居连接固定至少与其余1个12号结构单元连接。
- 13 `Carbocyclic_Aro_CH`（连接度=2）
  - 形式: 默认芳香周碳位点。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环）。
  - 规则: 不外接芳香结构。
- 10 `Aryl_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-芳香直连位点（芳基取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个4/10号结构，`4/10-10`成对连接，用于团簇间“刚性短桥”。
  - 上限: 单环最多 2；萘 3、菲 4、四环 6（你的上限规则）；总数保持偶数。
  - 规则: 固定外接10号结构。
- 11 `Alkyl_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-脂肪连接起点（烷基取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个脂肪结构`23/22/19/20/21/24/25`，用于团簇间“脂肪桥接”或作为脂肪侧链取代。
  - 规则: 固定外接1个脂肪结构；外接脂肪结构为19/20/21时，19/20/21需要满足各自的固定连接规则。
- 5 `O_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-氧连接起点（氧取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个含氧结构`2/28/29`，用于团簇间“氧醚键桥接”或作为酚取代。
  - 规则: 固定外接1个含氧结构单元，含氧结构仅限于`2/28/29`。
- 6 `N_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-氮连接起点（氮取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个氨基结构`0/27`，用于团簇间“氨基桥接”或作为氨基取代。
  - 规则: 固定外接1个氨基结构单元，氨基结构仅限于`0/27`（0号的氨基端）。
- 7 `S_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-硫连接起点（硫取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个硫基结构`31`，用于团簇间“硫醚桥接”或作为硫端基取代。
  - 规则: 固定外接1个硫结构单元，硫结构仅限于`31`。
- 8 `X_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-卤连接起点（卤取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个卤族结构`32`，作为卤族元素端基取代。
  - 规则: 固定外接1个卤结构单元，卤族结构仅限于`32`。
- 9 `Keto_Substituted_Aro_C`（连接度=3）
  - 形式: 芳香-羰基连接起点（羰基-C(=O)-取代）。
  - 可连: 连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），固定外接1个羰基结构`0/1/2/3`，用于团簇间“羰基桥接”或作为羰基取代。
  - 规则: 固定外接1个羰基结构单元，羰基结构仅限于`0/1/2/3`（0号的羰基侧）。

### C 类 不饱和链（14-18）
- 14 `Vinyllic_Cq`（连接度=3）
  - 形式: >C= 碳碳双键，双端桥接单元。
  - 可连: 不饱和端（双键端），按优先级 `15/16/14`；饱和端可连2个结构，按优先级 `23/24/25/22/19/20/21/2/1/0/3/4`（0/2号的羰基端），；
  - 规则: 不饱和端固定连接另一个同类结构的不饱和端，仅限于结构`14/15/16`，及不饱和双键结构成对存在；不可两端全为末端结构。
- 15 `Vinyllic_CH`（连接度=2）
  - 形式: -HC= 碳碳双键，双端桥接单元。
  - 可连: 不饱和端（双键端）可连 `15/16/14`；饱和端可连1个结构，按优先级 `23/24/25/22/19/20/21/2/1/0/3/4`（0/2号的羰基端）；
  - 规则: 不饱和端固定连接另一个同类结构的不饱和端，仅限于结构`14/15/16`，及不饱和双键结构成对存在；不可两端全为末端结构。
- 16 `Vinyllic_CH2`（连接度=1）
  - 形式: =CH2 碳碳双键，末端结构。
  - 可连: 不饱和端（双键端）可连 `15/14`。
  - 规则: 不饱和端固定连接另一个同类结构的不饱和端，仅限于结构`14/15`，及不饱和双键结构成对存在；作为末端结构，不可连接其余末端结构，不可`16-16`互连。 
- 17 `Alkynyl_Cq`（连接度=2）
  - 形式: -C(三键) 碳碳三键，双端桥接单元。
  - 可连: 不饱和端（三键端）可连 `17/18`；饱和端可连1个结构,按优先级 `23/24/25/22/19/20/21/2/1/0/3/4`（0/2号的羰基端）。
  - 规则: 不饱和端固定连接另一个同类结构的不饱和端，仅限于结构`17/18`，及不饱和三键结构成对存在；不可两端全为末端结构。
- 18 `Alkynyl_CH`（连接度=1）
  - 形式: (三键)CH 碳碳三键，末端结构。
  - 可连: 不饱和端（三键端）可连 `17`。
  - 规则: 不饱和端固定连接另一个同类结构的不饱和端，仅限于结构`17`，及不饱和三键结构成对存在；作为末端结构，不可连接其余末端结构，不可`18-18`互连。 

### D 类 脂肪碳（19-25）
- 22 `Alkyl_CH3`（连接度=1）
  - 形式: -CH3，末端结构。
  - 可连: 按优先级，`23/24/25/11/19/20/21/2/3/0/11/14/15/17`（0/2号的羰基端）。
  - 规则: 作为末端结构，不可连接其余末端结构，不可`22-22`互连。
- 23 `Alkyl_CH2`（连接度=2）
  - 形式: -CH2-，双端桥接单元。
  - 可连: 可桥接2个结构,按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 不可两端全为末端结构。
- 24 `Alkyl_CH`（连接度=3）
  - 形式: -CH<，双端桥接单元。
  - 可连: 可桥接3个结构，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 不可全为末端结构。
- 25 `Alkyl_Cq`（连接度=4）
  - 形式: >CH<，双端桥接单元。
  - 可连: 可桥接4个结构，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 不可全为末端结构。
- 19 `Alcohol_Ether_C`（连接度=2）
  - 形式: 脂肪碳-氧连接起点，形式固定为亚甲基-CH2-，双端桥接单元。
  - 可连: 可桥接2个结构单元，其中至少固定一端连接含氧结构，可连`2/28/29`（2号的醇端），另一端连接，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 至少存在一端连接含氧结构，含氧结构仅限于`2/28/29`（2号的醇端）；两端不可全为末端结构，29为末端结构。
- 20 `Amine_C`（连接度=2）
  - 形式: 脂肪碳-氮连接起点，形式固定为亚甲基-CH2-，双端桥接单元。
  - 可连: 可桥接2个结构单元，其中至少固定一端连接氨基结构，可连`0/27`（0号的氨基端），另一端连接，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 至少存在一端连接氨基结构，氨基结构仅限于`0/27`（0号的氨基端）；两端不可全为末端结构。
- 21 `Halogenated_C`（连接度=2）
  - 形式: 脂肪碳-卤组连接起点，形式固定为亚甲基-CH2-，双端桥接单元。
  - 可连: 可桥接2个结构单元，其中固定一端连接卤组结构，可连`32`，另一端连接，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`（0/2号的羰基端）。
  - 规则: 固定一端连接卤组结构，卤组结构仅限于`32`（0号的氨基端）；两端不可全为末端结构。

### E 类 杂原子节点（26-32）
- 26 `Heterocyclic_N`（连接度=2）
  - 形式: 芳香环内氮原子节点，固定为吡啶环中的氮。
  - 可连: 可环接2个芳香结构单元，按优先级`13/11/12/10/5/6/7/8/9`（成吡啶、吡咯环）。
  - 规则: 取代芳香环中的CH（13号结构）为杂原子氮；固定存在形式为吡啶中的氮；不外接其余结构。
- 27 `Amine_Nitrogen`（连接度=2/3 取决于 -NH2/-NH-）
  - 形式: 氨基结构，固定形式：-NH- 双端桥接单元；
  - 可连: 形式-NH-可桥接2个结构单元,固有连接`6/20`
  - 规则: 作为桥接单元不可两端同时连接末端；作为末端不能连接其余末端。
- 28 `Hydroxyl_O`（连接度=1）
  - 形式: 羟基，-OH，末端结构。
  - 可连: 可连接1个结构单元，固有连接`5/19`。
  - 规则: 作为末端不能连接其余末端。
- 29 `Ether_O`（连接度=2）
  - 形式: 醚键，-O-，双端桥接单元。
  - 可连: 可连接2个结构单元，固有连接`5/19`。
  - 规则: 作为桥接单元不可两端同时连接末端。
- 30 `Heterocyclic_S`（连接度=2）
  - 形式: 芳香环内硫原子节点，固定为噻吩环中的硫。
  - 可连: 可环接2个芳香结构单元，按优先级`13/11/12/10/5/6/7/8/9`（成噻吩环）。
  - 规则: 取代5元芳香环中的CH（13号结构）为杂原子硫；固定存在形式为噻吩中的氮；不外接其余结构。
- 31 `Thioether_S`（连接度=2）
  - 形式: 硫醚键，固定形式为-S-，双端桥接单元。
  - 可连: 可连接2个结构单元，固有连接`7/19`。
  - 规则: 作为桥接单元不可两端同时连接末端。
- 32 `Halogen_X`（连接度=1）
  - 形式: 卤组，-X，末端结构。
  - 可连: 可连接1个结构单元，固有连接`8/21`。
  - 规则: 作为末端不能连接其余末端。

接下来就是针对目前初始化修正之后的结构单元进行配置1-hop，我觉得应该这么做：首先是标注目前所有的结构单元，标注每个结构单元的唯一序号id（全局id）1,2,3,4，标注其结构单元类型id（例如32 `Halogen_X,32号，和唯一序号id区分），然后是连接度，然后是1-hop[- - -](例如13号，连接度是2,1-hop假设为[12 13]).然后需要分配所有结构单元的1-hop，分配逻辑如下：

先完成所有固定连接的分配：
a、32号X存在固定连接，连接度为1，固定分配1个对应的8号/21号，及32号的1-hop分配完成，后续无法更改；假设存在一个32号（全局id 300），layer0修正后存在1个8号（全局id 60），组合为（32，[8]，对应全局id 300，[60]）；注意所有的结构单元之间互为1-hop，对于8号（全局id 60）来说，32号（全局id 300）是8号的1-hop，及8，[- - 32](8号的连接度为3)，全局id 60，[- - 300]。

b、31号硫醚-S-存在固定连接，连接度为1，固定分配2个对应的7号、19号（先随机生成组合，例如（7、7/7、19/19、19），必须消耗所有存在的7号;例如存在1个31号（全局id 260），layer0修正后存在1个7号（全局id 80）和30个19号（全局id依次 161：190），组合为（31，[7 19]，对应全局id 260，[80 161]）,剩下29个19号，7号无），及31号的1-hop分配完成，后续无法更改；

c、0号的氨基端和27号存在固定连接，连接度分别为1/2，分别固定分配1个/2个对应的6号、20号（先随机生成组合，必须消耗所有存在的6、20号；例如1个0号，1个27号，layer0修正存在1个7号和2和20号，组合为（0，[20 -]（-表示待补充）;27，[6 20]）,剩下29个19号，7号无），及27号的1-hop分配完成，0号的1-hop未完成（连接度只完成1个），剩下的羰基端后续完成；后续可能更改。

d、2号中的氧羟基端，28和29号，连接度分别为1/1/2,固定连接1/1/2个5号、19号（先随机生成组合，必须消耗所有存在的5、19号（31号连接后剩下的19号））。及28号、29号的1-hop分配完成，2号的1-hop未完成（连接度只完成1个），剩下的羰基端后续完成；后续可能更改。

e、0号的羰基端、1号、2号的羰基端、3号分别的连接度为1/1/1/2，按优先级可连接1/1/1/2个 `9/22/23/24/25/14/15/17`，先随机生成组合，必须消耗所有存在的9号；这样0、1、2、3均完成全部的1-hop分配；后续可能更改。例如先前存在的一个0号，[20 -]和2号,[]

f、14、15、16号中的不饱和端的连接度为1/1/1，按优先级可连接1/1/1个 `15/16/14`，先随机生成组合；不饱和结构是两两配对的，且需要注意的是所有的结构单元之间互为1-hop。例如存在1个14号（全局id 100），2个15号（全局id依次 101：102）和1个16号（全局id 103）。两两组合，先14号，分配1个15号作为1-hop及[- - 15]，及全局id 100，[- - 101];然后是15号，分配1个16号作为1-hop及[- 16]，全局id 101，[- 103]。由于互为1-hop，对于15号全局id 101的1-hop为[- 14]及[- 100],对于16号全局id 103的1-hop为[15]及[101];这个配对是随机的，及14-14,14-14,14-16,15-15,15-16这些配对都是可以的
然后是17和18号的分配，由于layer0修正17、18一定刚好成对的，及17，[- 18];18，[17].注意连接度差异。

g、结构单元26和30号杂原子结构单元，连接度为2，按优先级`13/11/12/10/5/6/7/8/9`，优先合理且分散的分配空1-hop结构单元。

h、然后接下来就是完成固定连接结构的1-hop不完整结构的分配。例如先前的32号1-hop分配的8号，由于互为1-hop，目前这个8号的1-hop[- - 32]不完整！需要为其分配完整的1-hop，根据8号的连接度为3，还需要连接2个其余芳香结构单元,按优先级`13/12/11/10/5/6/7/8/9`（成芳香环），优先分配目前1-hop为空的其余的芳香结构单元，例如8号的1-hop分配完成[13 12 32]，假设全局id 60，[140  120 300], 全局id 140 120分别对应13号和12号,注意合理且分散的随机分配空1-hop芳香结构单元; 然后是7号与31号互为1-hop，分配完整的7号的1-hop，逻辑同8号；同理6号和5号以及9号。如果没有1-hop为空的其余的芳香结构单元，就分配其余1-hop不完整的芳香结构单元。例如，(8，[13 - 32])(全局id 60，[140 - 300]), 还缺一个芳香结构，但是没有1-hop为空的芳香结构，存在(9，[- - 3])(全局id 65，[- - 10])，那么就可以将这个9号分配至1-hop得到(8，[13 9 32])(全局id 60，[140 65 300])，同理对于这个9号由于互为1-hop，得到(9，[8 - 3])(全局id 65，[60 - 10]).需要注意互为1-hop，将某一个作为另一个结构单元的1-hop时，该结构单元的1-hop也包括了另一个结构单元！涉及信息的修正，为了防止仅少数几个结构单元就完全互为1-hop，因此鼓励优先使用1-hop为空的芳香结构单元。同理，对于21,20和19号，连接度都为2，与先前X、N、S、O互为1-hop之后已经分配1-hop中的1个及[- 32/31/29/28/27/0/2],需要完成没有分配的那一个，按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/16/17/18`，注意优先合理且分散的分配空1-hop结构单元。

i、然后是不饱和结构单元的分配,其中14,15,17的1-hop不完整，按照`23/24/25/22/19/20/21/2/1/0/3/4`，将1-hop分配完整，注意连接度差异，以及合理且分散的随机分配空1-hop结构单元。还有不饱和结构单元4号，连接度为1，按优先级`23/24/25/10`优先合理且分散的分配空1-hop结构单元。

j、接下来就只剩下10、11、12、13和22、23、24、25.先从22开始，其连接度为1，且部分22号因为结构单元互为1-hop已分配1-hop，因此需要将剩下的没有1-hop的进行分配；按优先级，`23/24/25/11/19/20/21/2/3/0/11/14/15/17`，优先合理且分散的分配空1-hop结构单元，如果没有空1-hop的，就补到不完整的1-hop结构单元中。其次是24（连接度为3）,25（连接度为4），同理按优先级`23/11/22/24/25/19/20/21/2/3/1/0/14/15/17`，针对目前24，25中存在的1-hop不全的结构，分配补全1-hop！注意事项一致！最后是 12、10、11和13依旧相互补全（可能都已经1-hop得到了部分的分配，但是不完整，需要相互成1-hop补完整！注意分散随机且合理！）

k、结构单元之间互为1-hop，例如23号和[22,11]互为1-hop，假设23号就这一个，与[22,11]互为1-hop之后，就不能成为其余结构单元的1-hop了！及该23号在1-hop的存在是有限的，假设该23号全局id是200，那么互为1-hop的22号和11号的1-hop就会包含该23号，包含这个全局id 200，23号的连接度为2，也就只会在1-hop中出现2次，其余的结构单元例如23,25就不能在1-hop使用该23号！尽可能的使用不同的结构单元，分散1-hop的组成类型，合理且随机，这里的优先级是强调的权重，而不是完全的按照全局id顺序和优先级依次分配全部高度一致的结构单元类型

最后，可能存在结构单元1-hop分配错误的问题，例如10,11，12,13或者22，23等结构单元数量限制！先基于我提供的逻辑完成所有结构单元1-hop的分配！目前还只是初始化的1-hop分配，后续需要基于谱图信息进行修正！请修正代码

非常好！接下来需要对目前的1-hop分配结果进行评估！统计每个节点及其1-hop对应的核磁信息！@z_library.py 中记录了众多的su+1-hop+2-hop的模版数据库subgraph_library.pt：
center_su_idx,center_su,hop1_multiset,hop2_multiset,sample_count,mu_min,mu_max,center_mu_median
0,Amide_Group,"[0,0,20]","[15,20,20]",5,151.0,151.375,151.375
0,Amide_Group,"[0,0,20]","[20,20,23]",3,150.875,150.875,150.875
0,Amide_Group,"[0,0]","[6,19]",3,154.375,154.625,154.625
0,Amide_Group,"[0,0]","[6,9]",3,157.25,157.375,157.375
0,Amide_Group,"[0,0]","[9,20]",4,155.625,159.0,156.375
可以基于当前的分配得到的su+1-hop，从数据库模版的获取su+1-hop对应的平均的核磁信息以及强度信息，然后将这些核磁进行洛伦兹函数扩峰，组合在一起生成完整的重构核磁谱图，再与当前的输入的目标核磁进行对比，以此评估当前1-hop分配的效果！不需要预测核磁！只需要使用数据库的评估核磁进行评估即可！在目前的代码layer1上增加评估！ 

首先我需要你帮我整理一下目前的子图模版数据库！目前的子图模版不符合我先前定义的连接规则！我需要你写一份代码针对性的整理子图（暂时只需要整理1-hop即可）。例如：
center_su_idx,center_su,hop1_multiset,hop2_multiset,sample_count,mu_min,mu_max,center_mu_median
0,Amide_Group,"[0,0,20]","[15,20,20]",5,151.0,151.375,151.375
0,Amide_Group,"[0,0,20]","[20,20,23]",3,150.875,150.875,150.875
0,Amide_Group,"[0,0]","[6,19]",3,154.375,154.625,154.625
0,Amide_Group,"[0,0]","[6,9]",3,157.25,157.375,157.375
我们定义的规则里面，0号： C=O 端可连，按优先级 `9/22/23/24/25/14/15/17`；N 端固定可连 `6/20`；所以1-hop存在的[0,0,20] [0,0]就是完全不正确的！需要筛选满足我定义的连接规则的1-hop模版！注意需要包括连接度和连接组合，例如0号，连接度为2，组合需要符合[9/22/23/24/25/14/15/17,6/20]，类似的对于5号，连接度为3，需要满足组合[13/12/11/10/5/6/7/8/9,13/12/11/10/5/6/7/8/9,2/28/29]。我需要你在现在的已经生成的子图数据库的基础上进行提取和筛选，生成新的子图数据库！

以下是各结构单元的连接度和1-hop连接单元组合：
- 0 : 连接度为2，[9/23/24/25/22/14/15/17,6/20]
- 1 : 连接度为1，[9/23/24/25/19/20/21/14/15/17]
- 2 : 连接度为2，[9/23/24/25/22/19/20/21/14/15/17,5/19]
- 3 : 连接度为2，[9/23/24/25/22/19/20/21/14/15/17,9/23/24/25/19/20/21/14/15/17]
- 4 : 连接度为1，[23/24/25/10]
- 5 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，2/28/29]。
- 6 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，0/27]。
- 7 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，31]。
- 8 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，32]。
- 9 ：连接度为3，[13/12/11/10/5/6/7/8/26/30，13/12/11/10/5/6/7/8/26/30，0/1/2/3]。
- 10 ：连接度为3，[13/12/11/5/6/7/8/9/26/30，13/12/11/5/6/7/8/9/26/30，4/10]。
- 11 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，23/24/25/22/19/20/21/]。
- 12 ：连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30，12]。
- 13 : 连接度为3，[13/12/11/10/5/6/7/8/9/26/30，13/12/11/10/5/6/7/8/9/26/30]。
- 14 : 连接度为3，[23/24/25/22/19/20/21/2/1/0/3/4，23/24/25/22/19/20/21/2/1/0/3/4，14/15/16]。
- 15 : 连接度为2，[23/24/25/22/19/20/21/2/1/0/3/4，14/15/16]。
- 16 : 连接度为1，[14/15]。
- 17 : 连接度为2，[23/24/25/19/20/21/2/0/3，17/18]。
- 18 : 连接度为1，[17]。
- 19 ：连接度为2，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17，2/28/29/31]。
- 20 : 连接度为2，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17，0/27]。
- 21 ：连接度为2，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17，32]。
- 22 : 连接度为1，[23/11/24/25/19/20/21/2/3/1/0/14/15/17]。
- 23 : 连接度为2，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17]。
- 24 : 连接度为3，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17]。
- 25 : 连接度为4，[23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17,23/11/22/24/25/19/20/21/2/3/1/0/14/15/17]。
- 26 : 连接度为2，[13/12/11/10/5/6/7/8/9,13/12/11/10/5/6/7/8/9]。
- 27 : 连接度为2，[6/20,6/20]。
- 28 : 连接度为1，[5/19]。
- 29 : 连接度为2，[5/19,5/19]。
- 30 : 连接度为2，[13/12/11/10/5/6/7/8/9,13/12/11/10/5/6/7/8/9]。
- 31 : 连接度为2，[7/19,7/19]。
- 32 : 连接度为1，[8/21]。

接下来需要针对性的修正目前的1-hop分配以及初始的结构单元分布！修正的逻辑如下：

首先调整1-hop的分配。可以结合计算得到的差谱图，找到其中的正峰和负峰的位置，然后从目前的分配好的结构单元+1-hop表中找出处于这个核磁位置是哪些结构单元！结合结构单元的核磁分布范围su_nmr_common_range_filtered.csv进行调整。先找到差谱图中为正峰和负峰的位置，然后从目前的分配好的结构单元+1-hop表中找出处于这个核磁位置是哪些结构单元！

1、首先修正结构单元的分布！从筛选得到的su_nmr_common_range_filtered.csv中获取各个结构单元的核磁范围，结合layer1_library_spectrum_comparison.csv中的差谱图来调整结构单元。调整逻辑如下：找差谱图中的负峰，负峰表示对应的结构单元1-hop过于集中！假设大量13,Carbocyclic_Aro_CH,[9 11]，127.75；13,Carbocyclic_Aro_CH,[10 11],127.3；11,Alkyl_Substituted_Aro_C,[23 23 25]，127.3；12,Aromatic_Bridgehead_C,[10 12 13],126.625这部分叠加在一起之后生成一个大尖峰，124-129范围内对应形成差谱图中负峰！然后从差谱图中找到相邻范围内的正峰（结合su_nmr_common_range_filtered.csv），正峰表示缺乏对应的结构！假设在130-136、114-119范围存在大正峰，然后从layer1_library_node_peaks.csv SU+1-hop模版中找能够填补123-120范围的正峰对应的结构单元！例如13,Carbocyclic_Aro_CH,"[5,26]",133.75，12,Aromatic_Bridgehead_C,"[5,10,12]",115.625该模版的中心基本能够填补130-136，114-119范围的正峰！那么部分结构单元就应该将13 [10 13]（假设全局 id 185 [130 150]）（或13 [10 11](假设全局 id 186 [131 140])）为调整13 [5 26]! 需要注意的是调整涉及到其余结构单元的相应调整，还有注意的是调整的数量基于存在结构单元的数量，比如26只有1个，连接度为2，已经形成了26 [9 12]（假设全局 id 255 [100 145]）的结构，前面提到可以转化得13号的1-hop得到13 [5 26]（可以得到两个，对应26号的两个连接度），那么26 [9 12]就变为26 [13 13]（全局id 255 [185 186]）。由于互为1-hop，那么原本13 [10 13]（全局 id 185 [130 150]）中互为1-hop的10 [13 -]（id 130 [185 -]） 和 13 [185 -](id 150 [185 -])中的13（id 185）就会被26 [9 12]（id 255 [100 145]）中的9号/12号（id 100/145）替换！同样的，原本13 [10 11](全局 id 186 [131 140])中互为1-hop的10号和11号中的1-hop包含了13号会被26 [9 12]中的12号/9号替换！（上一个用9号换13号，这个就用剩下的12号去换13号）！这就是基本的更换逻辑！需要遵守结构单元原本的连接度和连接规则，才能替换（数据库中的1-hop都是遵守的）！可以针对不同结构单元的类型进行分开调整，例如先调整芳香结构，然后调整羧基结构，调整非饱和结构，最后调整脂肪碳结构等等！此外，调整可能无效，部分结构单元的1-hop无论怎么调整可能都无法消除负峰，补全正峰，这不重要，只需要尽可能的调整能够调整的部分即可！

需要分区域进行1-hop的修正调整！目前的1-hop调整程度明显不足。应该分区域0-90，90-160，160-240依次进行1-hop的调整，每部分均分别进行1-hop的调整！

### layer2：2-hop的分配和调整

前面已经完成了1-hop的分配和调整，接下来进行2-hop的分配，给当前所有的结构单元分配2-hop。实际上在确定1-hop的时候，就已经确定了2-hop的结果！例如假设存在以下结构单元：22 [23]（全局id A [B]）;23 [22 11]（全局id B [A C]）;11 [5 23 13]（全局id C [D B E]); 13 [11 13]（全局id E [C F])；13 [13 13]（全局id F [E H])。那么这里面22号（id A）的2-hop就是[11]([C]);23号（id B）的2-hop就是[5 13]([D E]); 13号（id E）的2-hop就是[5 23 13]([D B H])。就是说想要得到一个结构单元的2-hop，需要找到该结构单元的1-hop，基于结构单元之间互为1-hop，然后找到1-hop里面各个结构单元的1-hop即可（1-hop的1-hop中的其余结构单元，不包括自身结构单元）。就像案例中的13 [11 13]（全局id E [C F])，其中13号的1-hop是11号C和另一个13号F，分别得到这两个结构单元的1-hop，及11 [5 23 13]（全局id C [D B E])和13 [13 13]（全局id F [E H])，组合这些1-hop，得到[5 23 13 13 13](D B E E H)去掉互为1-hop中的本身13号E，就得到[5 23 13]([D B H])，这就是13号E的2-hop！

首先计算出所有结构单元的2-hop！这样就完成了2-hop的分配！及构建得到了完整的模版。然后进一步的需要从数据库中找到对应的模版，然后对应模版需要分配环境潜变量z，初始分配的z是结构单元+1-hop+2-hop对应的中位的代表的z！可能存在2-hop模版在数据库中找不到的情况，那么就找最符合当前2-hop组成的模版。例如示例的13 [11 13] [5 23 13]这个模版。其中2-hop[5 23 13]在数据库中没有！调用G2S模型并使用这个z来进行核磁的预测。

如何在不存在2-hop模版的情况下，匹配最近的2-hop模版？逻辑如下:首先1-hop基本上都是存在的，重点是2-hop！2-hop找最近首先是节点组成最相似的，假设2-hop为[A A B C D]，但在模版库中找不到，找最相似的就是按照重合度来排列，假设存在2-hop模版为[A B B C D]/[A A B D D],与[A A B C D]的重合度为相差1个节点，这些是优先级第一档可以近似选取的模版（及与当前的2-hop只相差1个节点），那么选哪个模版？不是按照L1 距离计算，结构单元5号，13号这些都是结构单元类型id，作为距离计算没有意义。因此，应该优先选第一档的2-hop近似模版中化学位移最接近当前该2-hop对应的1-hop的中位化学位移的！及完整的模版是A [A B D] [A A B C D](中心SU+1-hop+2-hop)，但是当前的2-hop不存在对应的模版！其中在layer1中中心SU+1-hop做核磁评估的时候取了1-hop的处于中位数的核磁化学位移。对此，在寻找2-hop的近似模版中优先选与1-hop的处于中位数的核磁化学位移最接近的模版！（在第一档存在多个模版的情况下！从第一档2-hop模版中选最接近1-hop中位化学位移的模版）！假设不存在第一档的2-hop模版，那么就选第二档的2-hop模版，即与当前2-hop相差2个节点的模版！同样，也是从第二档模版选取与当前的1-hop的中位化学位移最接近的模版！）例如，当前的完整的模版是A [A B D] [A A B C D]，其中1-hop使用的核磁是130ppm，但是没有2-hop模版，发现近似模版A [A B D] [A B B C D]以及A [A B D] [A A B D D]，对应的核磁分别为136ppm和128ppm，那么就应该选取第二个模版，及A [A B D] [A A B D D]这个！

### layer3:环境潜变量的调整

layer2完成了2-hop的计算以及初始z的分配。现在需要对z的选取进行优化。完整的模版SU+1-hop+2-hop是包括多个不同的环境潜变量z的（及完整3-hop子图组成存在差异），需要结合差谱图做进一步的细选和调整！例如：
center_su_idx,center_su,hop1_multiset,hop2_multiset,sample_count,mu_min,mu_max,center_mu_median
0,Amide_Group,"[14,20]","[0,1,15,23]",4,166.375,166.375,166.375
12,Aromatic_Bridgehead_C,"[12,13,26]","[0,12,13,13,13]",26,136.125,137.625,136.75
12,Aromatic_Bridgehead_C,"[12,13,26]","[0,12,13,13]",70,134.75,139.5,136.0
22,Alkyl_CH3,[14],"[16,23]",28,22.44,23.25,22.60
22,Alkyl_CH3,[14],"[16,24]",114,19.99,21.234375,20.65
调整逻辑如下:
a、首先基于差谱图明确正峰和负峰的位置，负峰表示结构单元的核磁累积过多，正峰则表示过少。因此需要将出于负峰位置的结构单元模版对应的z向正峰方向微调，当前选取的z都是SU+1-hop+2-hop对应的中位/平均值，如果此时该结构单元的核磁处于负峰位置（假设负峰处于126ppm），对应环境潜变量z1，发现在120左右就存在正峰，该模版的环境潜变量z对应核磁范围包括120ppm，那么就将z1替换为z2，z2对应的核磁在120ppm位置，及实现了核磁的调整，使得核磁峰分布更加接近目标核磁！这一步只是从模版数据中选取更好的z。

可以分为多个区域进行！按照0-90,90-160,160-240三个区域！优先从0-90区域开始！确定最大负峰，然后附近的正峰，找到负峰对应的所有结构单元（对应的结构单元类型及其全局id）！按照优先级:22、23、24、25、19、20、21的顺序进行z的微调！先选1个结构单元，同类结构单元从全局id小的开始！查找该结构单元对应的模版的范围是否能够填补附近的正峰！若能选取模版中新的z去填补正峰！不能则不进行调整！不调整的结构单元按照全局id和结构单元类型优先级顺位到下一个结构单元，评估其模版中是否存在z能够填补正峰！例如评估差谱，发现0-90ppm中在29ppm处存在最大负峰，按照优先级找到1个结构单元23号全局id 250，该结构单元的z范围覆盖包括25-32ppm，发现在25ppm存在最大正峰（找该结构单元z范围内能够填补的最大正峰），调整该结构单元的z。然后继续评估0-90ppm，在29ppm处还是存在负峰，按照优先级找到1个结构单元23号全局id 251，该结构单元的z范围覆盖包括25-32ppm，还是发现25ppm存在最大正峰，调整该结构单元的z。再次继续评估0-90ppm，在29ppm处还是存在负峰，按照优先级找到1个结构单元23号全局id 252，该结构单元的z范围覆盖包括25-32ppm，发现此时32ppm存在最大正峰，调整该结构单元的z。继续评估0-90ppm，在29ppm处还是存在负峰，按照优先级找到1个结构单元23号全局id 256，发现该结构单元的z范围内不存在正峰，不进行调整，随后继续按照优先级找到结构单元25号全局id 285，z范围24-38ppm，能够填补的最大正峰35ppm，调整该结构单元的z。

然后是90-160区域，同样确定最大负峰，然后附近的正峰，找到负峰对应的所有结构单元！与0-90的逻辑一致！优先级:13,12,11,10,9,5,6,7,8,14,15,16,17,18,4;最后是160-240区域，同理，优先级为1,2,3,0.注意，每个区域分别进行评估和调整，每次调整作为1个小循环，及但凡调整之后能够使得单个区域的核磁峰分布更加接近目标核磁就接受调整，直到单个区域的核磁峰分布无法再进行调整，即完成单个区域的调整！

b、调整之后还存在正负峰，就需要找到负峰对应的结构单元里面的2-hop模版采用的是近似模板，而不是真实模板的！及假设一个完整的模版SU+1-hop+2-hop其中2-hop实际上在2-hop模板数据库种是不存在的，采用了近似模板，现在就需要进行近似模板的更替了！再进行前面的分区域z微调后还是存在负峰，说明结构单元依旧过多！但是已经无法再进一步调整当前的z了。因此这里需要进行近似2-hop模板的调整！分0-90,90-160,160-240三个区域进行调整！首先还是确定负峰的位置对应那些结构单元，再确定这些结构单元里面那些采用了近似2-hop模版！还是按照和先前一样的优先级！

例如评估差谱，发现0-90ppm中在29ppm处存在最大负峰，按照优先级找到1个结构单元23号全局id 260，该结构单元的2-hop采用了近似2-hop模板（当前的正确的模版是A [A B D] [A A B C D]，但是2-hop不存在，找到近似模板A [A B D] [A B B C D]），依旧按照重合度为相差1个节点为优先级第一档，分档2-hop近似模板。发现存在正峰22ppm，其中恰好第一档2-hop模板中的[A A B D D]的z覆盖范围覆盖了22ppm，然后调整近似模板A [A B D] [A B B C D]为A [A B D] [A A B D D]，并选取此模板的z为对应22ppm的z！然后继续评估0-90ppm，在29ppm处还是存在负峰，继续查找使用近似2-hop模版的结构单元，按照优先级找到1个结构单元，继续尝试更换近似模板，来覆盖存在的正峰！同类，对于90-160,160-240也是如此！每个区域分别进行评估和调整，每次调整作为1个小循环，及但凡调整之后能够使得单个区域的核磁峰分布更加接近目标核磁就接受调整，直到单个区域的核磁峰分布无法再进行调整，即完成单个区域的调整！

### layer4：基于差谱图结构单元的调整

2、1-hop的装配显著受到初始的结构单元分布的影响！因此还需要调整初始结构单元的分布！调整逻辑如下：

a、先调整碳基类结构分布！根据目前重构的核磁谱图，我们能够得到160-240部分的核磁差谱图。然后根据su_nmr_common_range_filtered.csv中的羰基结构的统计核磁范围，找到差谱图中对应的峰，这里给出了4个羰基结构的核磁峰的中位数：0,Amide_Group,167.125；1,Carboxylic_Acid,174.8；2,Ester_Group,169.6；3,Aldehyde_Ketone_C,195.8 ；然后对应的在差谱图中计算对应范围是否存在正峰或者负峰，正峰表示缺乏结构负峰表示结构过多；例如1,Carboxylic_Acid,174.8需要审查在174.8+-5ppm范围的正负峰分布情况，如果为正峰，则需要增加1号结构，根据羰基结构守恒，必然在160-240ppm的其他位置存在负峰（存在负峰不显著的情况，3号羰基的峰中心位置分布较宽），通常是3号结构（195.8ppm），因此需要减少3号结构！同理对于2号结构也是这样，审查169.6+-5ppm范围的正负分布情况。对于范围内正负峰皆有的情况，且正负峰不是大尖峰的情况，可以视为该范围结构正确！注意这里只调整1,2,3号结构单元，固定0号结构不做修改！也就是说，存在1-3,2-3（1-2适度转化，优先转化3号）和1-2的相互转化！要求转化中，至少保留一个结构单元（例如1号的正峰面积与2,3的差异峰面积相比，1号的面积占据绝对比例，3号几乎没有，2号正负几乎相抵，那么3号转为1号，至少保留1个3号！相互转化之后存在O和H的变化，需要重新进行O、H的修正！及layer0修正O和H的逻辑！修正之后重新计算羰基类结构固定连接9号，醚固定连接5号和19号的数量。然后重新分配1-hop！计算核磁谱图，调整1-hop！

流程应该是这样的:初始的结构单元分布及其修正，然后layer1的1-hop初始化分配，然后1-hop的修正！然后基于差谱评估对 H_final 做1/2/3 互转调整！ 然后回到结构单元中修正部分，修正羰基之后，然后重新计算当前的O含量及修正28号和29号（@inverse_pipeline_v3.py 代码中layer0部分对应28、29号的修正！）！修正之后，还需要修正羰基的固定连接结构9号的数量！再修正28,29的固定连接5、19号的数量！然后再进行C、H的修正，及修正脂肪碳，非饱和结构以及芳香碳，最后的H的调整！这样就完成了整个结构单元分布的修正，这是基于差谱图进行的修正。然后，进行layer1-2-3，及1-hop，2-hop和z的微调！然后再基于差谱图评估 H_final 做1/2/3 互转调整，再次循环！

b、在调整羰基类结构之后，并循环完成layer0-1的调整，然后进行9号的调整。初始的9号是按照C=O的连接数量取60%，但是0/1/2/3和9号的连接可能过多或过少，导致无法匹配目标核磁峰！例如假设核磁差谱峰存在173ppm的正峰，但是已经完成了羰基结构的修正及对应的羰基结构如1号是充足的，但是1号的核磁位移却不在这里，1-hop的调整也没有效果！结合su_hop1_nmr_range_filtered.csv中发现1,Carboxylic_Acid,[9],169.375与173ppm不匹配！1,Carboxylic_Acid,[19],172.125与173ppm匹配，但是目前的结构单元组成存在较多的9号，导致1号大量与9号连接（规定C=O必须消耗所有的结构单元），导致其在1-hop调整中是无法与19号连接。这就表明，需要减少9号的数量！初始设置的按照C=O的连接数量取60%导致9号过多！对此就需要减少9号的数量！由于9号属于芳香结构单元，需要满足于芳香结构数的守恒，因此9号需要转为11/13/12号等芳香结构单元，不是直接删去9号，而是进行转化！然后循环完成layer0-1，9号转化为13号可能引起H的增加，需要修正H的修正！然后初始化1-hop分配再调整1-hop！反之，如果差谱图中在169ppm部分存在正峰，表示缺少9号结构单元，则需要将11/13/12号等芳香结构单元转化为9号结构单元，然后修正H，初始化1-hop分配再调整1-hop！

逻辑应该是这样的，在羰基修正之后，运行layer0-1之后，得到了评估核磁的结果，观测160-240是否存在的正负峰，找到关键峰位，然后针对峰位看其对应的结构单元是1,2,3中的哪一类，例如核磁差谱峰存在173ppm的正峰，此时已经无法调整羰基了，需要调整的只能是9号！9号调整之后，1,2,3号就减少了与9号的固定连接，增加与23/24/25/19/20/21/14/15/17的连接！例如1,Carboxylic_Acid,[19],172；1,Carboxylic_Acid,[20],176；1,Carboxylic_Acid,[21],173；1,Carboxylic_Acid,[23],177；1,Carboxylic_Acid,[24],1751；Carboxylic_Acid,[9],169。明显的1 [9]的组合无法填补173ppm的峰位，如果1 [19]就可以！因此通过增减9号，调整1,2,3与其他结构的连接就能够进行的差异峰的修复！这才是正确的逻辑！

流程：完成羰基结构的调整后，进行一次循环完成layer1-2-3的调整，得到最终调整后的差谱图；然后基于这个差谱图进行9号的调整，再循环完成layer1-2-3的调整。不是进行多个羰基结构的调整循环，而是单次羰基结构的调整并完成一次layer0-1循环之后，评估得到差谱图，基于差谱图进行9号的调整！9号调整后，再循环完成layer0-1！这两部分组成一个大循环！然后继续重复大循环，保留最好的结果，直到结果无法进步一改善。

c、然后进行O的调整，前面的修正调整了氧（氧醚键）结构单元2，28，29号，这里杂原子结构单元分布就不在进行调整！这里需要修正的是其固定连接5号和19号的分布。layer0的5/19分布是按照优先级为19>5，比例为0.55:0.45进行修正的，是人为定义的结构，这里需要基于差谱进行正确的分布调整。5号,19号化学位移的中位数是：5,O_Substituted_Aro_C,154.75；19，Alcohol_Ether_C,66.6。同样的，需要针对性的审查对应范围内的正负峰分布。如果5号对应的差谱图为正峰，说明缺乏这部分，需要增加；反之如果差谱图为负峰，说明过剩这部分，需要减少。若5和19同为正峰或负峰，则针对差谱峰更大的，对5/19号进行调整，例如同为负峰，19号的负峰大，+5/-19，若5号的负峰大，则+19/-5。假设5号的差谱图为正峰且较大，则增加5号的数量，对应的19号的数量减少！但是不能直接+5号，-19号，会引起芳香结构和脂肪结构的失衡！因此增加5号时，是将13/11号转为5号！减少19号，是将19号转为23号！这个还是引起H的变化.因此还需要进行H的修正！修正之后才是运行layer1的1-hop分配以及1-hop的调整！需要注意的是这里对O的固定连接19号的数量调整是不包括S的31号硫醚对应的19号！O通常含量较多，不允许5,19号出现单边为0的情况！

然后进行N氨基类结构固定连接的调整，及对6号和20号的数量调整！逻辑与调整O的固定连接类似！20,Amine_C,49.375；6,N_Substituted_Aro_C,146.375；同样的，需要针对性的审查对应范围内的正负峰分布。如果6号对应的差谱图为正峰，说明缺乏这部分，需要增加；反之如果差谱图为负峰，说明过剩这部分，需要减少。若6和20同为正峰或负峰，则针对差谱峰更大的，对6/20号进行调整，例如同为负峰，20号的负峰大，+6/-20，若6号的负峰大，则+20/-6。假设6号的差谱图为正峰且较大，则增加6号的数量，对应的20号的数量减少！但是不能直接+6号，-20号，会引起芳香结构和脂肪结构的失衡！因此增加6号时，是将13/11号转为6号！减少20号，是将20号转为23号！这个还是引起H的变化.因此还需要进行H的修正！修正之后才是运行layer1的1-hop分配以及1-hop的调整！N通常含量较少，允许6,20号出现单边为0的情况！

然后是S硫醚类的固定连接7号和19号的调整！逻辑与调整O的固定连接类似！注意这里的19号只包括了31号硫醚连接的19号，O连接的需要排除；7,S_Substituted_Aro_C,152.875；19，Alcohol_Ether_C,66.6；同样的，需要针对性的审查对应范围内的正负峰分布。如果7号对应的差谱图为正峰，说明缺乏这部分，需要增加；反之如果差谱图为负峰，说明过剩这部分，需要减少。若7和19同为正峰或负峰，则针对差谱峰更大的，对7/19号进行调整，例如同为负峰，19号的负峰大，+7/-19，若7号的负峰大，则+19/-7。假设7号的差谱图为正峰且较大，则增加7号的数量，对应的19号的数量减少！但是不能直接+7号，-19号，会引起芳香结构和脂肪结构的失衡！因此增加7号时，是将13/11号转为7号！减少19号，是将19号转为23号！这个还是引起H的变化.因此还需要进行H的修正！修正之后才是运行layer1的1-hop分配以及1-hop的调整！

对于X的32号，其固定连接结构为8号和21号，su_nmr_common_range_filtered.csv中统计的核磁范围21,Halogenated_C,38.4；8,X_Substituted_Aro_C,131.4；然后找对应范围的差谱图，如果21号的差谱图为正峰，说明缺乏这部分，需要增加；反之如果差谱图为负峰，说明过剩这部分，需要减少。若8和21同为正峰或负峰，则针对差谱峰更大的，对8/21号进行调整，例如同为负峰，21号的负峰大，+8/-21，若8号的负峰大，则+21/-8。假设8号的差谱图为正峰且较大，则增加8号的数量，对应的21号的数量减少！但是不能直接+8号，-21号，会引起芳香结构和脂肪结构的失衡！因此增加8号时，是将13号转为8号！减少21号，是将21号转为23号！相反，8号的差谱图为负峰且较大，则8号转为13号，23号转为21号。允许8/21号单边为0的情况！但是必须满足32所需的8/21号的总数守守恒！


e、基于差谱和资源分配耦合的结构单元调整！layer4部分需要耦合资源分配的逻辑来评估当前结构单元分布的合理性！（注意这里进行的资源分配评估是使用转化之后的资源进行分配，及将5,6,7,8,9,11全部视为11号结构单元！0,2,3,15,17,19,20,21,23,27,29,31全部转为23号，1，4,16,18,22,28，32全部转为22号，14、24全部转为24号，但是后续的结构单元调整是只调整11,12,13,22,23,24,25，对于被转化的其他结构单元不进行调整！因为被转化的结构单元在前面的循环中已经完成了调整，最后这里调整的都是骨架级结构单元！）在不满足资源分配的情况下，优先通过调整结构单元的分布来满足资源分配的要求，在满足资源分配的情况下，才基于差谱进行结构单元分布的调整！

第一、首先基于当前的结构单元分布，进行资源分配的评估，如果分配资源时候发现22号结构单元不足，而24/25分支结构单元过剩，表示形式了24/25号的拓扑结构无法封端，及无法完成24/25分支结构的资源分配（22号是末端结构单元，24/25是中间连接结构单元），处理逻辑如下：在满足H总数量偏差在±3%以内的情况下，多次循环通过同时减少分支24/25号增加22号，直到能够实现资源的分配！及假设当前22号结构单元只有a个，而24号/25号结构单元存在b个，无法完全实现结构单元的分配！则每次减少1个25号和24号结构单元，增加2个22号结构单元，25号结构单元数量最多减少到脂肪碳总数的2%（如果25号已经减少到总数的2%，则每次减少2个24号，增加2个22号），然后再次尝试资源分配，如果能够正确实现24/25的分配，则结束调整，否则循环上述过程，直到能够正确实现24/25的分配。及通过循环调整22\24/25实现分支结构单元24/25的正确分配！循环调整中会导致H增加，如果出现H的数量超出了+3%的范围，则可以优先依次循环将一对14/15/16（14/15/16成对转化，如14-15,15-16,15-15，但是注意16的数量不能超过14/15的总和，且14/15/16的总数低于5%的13号时就停止14/15/16的转化）转化为1个12和1个13，然后尝试将13号转为12号结构单元或者23号转为13号结构单元来实现H的减少！          此外，以下两种情况也可能存在：1.可能存在11号不足导致无法实现分支24/25分配的情况，具体表现为，用于配置分支连接的11不够，例如脂肪环/并环/单链都需要11号来完成组成（代码里面有每个分支类型，比如脂肪环需要多少的11号，23号以及22号），但是发现不足，此时依次循环将12/13号转为11号即可，满足分支的分配（分支需要的11号较少）！2.还可能23号不足的情况，具体表现为脂肪环/并环/单链都需要23号来实现拓扑组装（类似需要22号来封端），依次进行13号转为23号（增加H，若H的数量超出了+3%的范围，尝试将一对14/15/16（14/15/16成对转化，如14-15,15-16,15-15，但是注意16的数量不能超过14/15的总和）转化为1个12和1个13，，若14/15/16的总数低于5%的13号时就停止转换，或者转化的数量超过13号总数的10%，也停止转化）以及同时将1个22号+1个24/25号转为2个23号来增加23号的结构单元数量！直到能够正确满足分支的分配！

第二、然后继续评估11号结构单元的数量在资源分配中是否合理，表现形式为资源分配中存在超过5个的11-23-11的连接方式（必须固定连接方式进行资源分配过后，剩下资源分配了大量的11-23-11，及extra类型的，不能是固定连接类型），需要减少11号的数量，通过将11号依次转为13号/12号结构单元来实现，减少到资源分配中不存在超过5个的11-23-11的连接方式！注意这个过程中会增加H的数量，如果增加的H的数量超出了+3%的范围，则不再进行11号的调整！相反，如果资源分配时发现11号结构单元的数量过少，例如存在超过5条长度为6个23号的柔性桥接链或者脂肪侧链（必须固定连接方式进行资源分配过后，剩下资源分配了大量的6个23号的柔性桥接链或者脂肪侧链，及extra类型的，不能是固定连接类型），则需要适当增加11号，通过依次循环将13号转为11号或者12号转为11号来实现！注意这个过程中会减少H的数量，如果减少的H的数量超出了-3%的范围，则不再进行11号的调整！       每次11号调整之后继续执行资源的分配，实现合理的资源的分配！

注意上诉的调整过程，执行调整之后需要立刻重新进行资源的分配评估，直到能够实现合理的资源的分配！且满足元素H分布在3%以内！然后才开始循环layer1-2-3！循环完成后，评估核磁，此时由于已经满足了资源的合理的分配，还需要进一步基于差谱图尝试进行结构单元的调整，来实现最优的结构单元分布！和其余阶段的结构单元调整逻辑类似，基于范围的差谱图（实际上就是12/13和24\25\22号的调整），如果12/13和24\25\22号的差谱图为正峰，说明缺乏这部分，需要增加；反之如果12/13和24\25\22号的差谱图为负峰，说明过剩这部分，需要减少。例如，每次调整限制为增加一个12号对应减少一个13号，减少2个23号对应增加1个22号和1个24/25号,注意25号不得超过脂肪碳总数的3%。调整之后，还需要进行资源分配评估以及不能超过H总数量偏差的±3%，如果资源分配不合理，需要调整，调整之后再次循环layer1-2-3.及首次进行layer4的最后一个阶段时，先进行资源分配评估，首次进入这个阶段不需要利用核磁评估的信息，然后循环layer1-2-3！       此后进入正式的基于差谱和资源分配耦合的结构单元调整阶段，设置循环。及layer3之后再进行核磁评估，基于核磁评估调整结构单元，然后再进行资源分配的评估（上诉的第一和第二评估内容），如果不符合资源分配，进一步调整结构单元，然后再次循环layer1-2-3！

#### 分支调整
需要调整一下基于资源分配调整结构单元的逻辑，layer4部分需要资源分配的逻辑来评估当前结构单元分布的合理性！（注意这里进行的资源分配评估是使用转化之后的资源进行分配，及将5,6,7,8,9,11全部视为11号结构单元！0,2,3,15,17,19,20,21,23,27,29,31全部转为23号，1，4,16,18,22,28，32全部转为22号，14、24全部转为24号（资源分配中Phase 3b: 支链24/25分类，对于14号转为24号的，这类14号与24号类似的基于1-hop信息分为ABCD四类），但是后续的结构单元调整是只调整11,12,13,22,23,24,25，对于被转化的其他结构单元不进行调整！因为被转化的结构单元在前面的循环中已经完成了调整，最后这里调整的都是骨架级结构单元！）在不满足资源分配的情况下，优先通过调整结构单元的分布来满足资源分配的要求！

正确的逻辑应该是这样的：首先是分支分配的检查！通过资源分配评估当前的分支是否能够正确分配，已经完成资源分配Phase 4.5: 支链分配 (_allocate_branches)之前的任务。例如假设目前还存在2个25号，20个23号，4个22号，20个11号，按照逻辑应该先开始Step 0: SU 25 优先分配，假设这里的25号的连接形式11-25-23-23-22 + 22 + 22（按照资源分配的逻辑来定连接形式），存在两条这样的25号的连接路径，及需要消耗2个25，6个22号，2个11号和4个23号，明显的，目前的22号不足，欠缺2个22号，对于这种欠缺22号数量少于3的情况，可以考虑将对应欠缺数量的23号转化为22号来补充22号的数量，这样会增加H，需要进行H的调整（策略放在后面），然后进行25号的资源分配评估（Step 0: SU 25，及实现25号的资源分配，能够分配则通过，反之不通过），如果通过则进行后续的资源分配阶段评估！对于欠缺数量超过3个的情况，将25号转为24号（如果25号的数量本身不为0，且目前少于脂肪碳总数的1%,则不转化；至少保留1个25号），确保剩下的25号能够正确的进行资源分配！注意25号转化24号是一次转化1个，然后进行25号的资源分配评估（及当前的25号能够实现资源分配），如果通过则进行后续的资源分配阶段评估，如果不通过则继续转化1个25号为24号，直到25号通过资源分配评估！不是一次性的转化！

假设目前还存在2个25号和5个24号，20个23号，7个22号，20个11号，按照逻辑应该先开始Step 0: SU 25 优先分配，假设这里的25号的连接形式11-25-23-23-22 + 22 + 22，存在两条这样的25号的连接路径，及需要消耗2个25，6个22号，2个11号和4个23号，能够完成资源的分配目前剩下的资源为：0个25号，5个24号，14个23号，1个22号，18个11号！然后就是24号的分支，计算目前所有的24号能够匹配那些分支（及对于当前存在的24号，需要计算其能够匹配那些分支类型来消耗所有的24号），假设5个24中存在3个C类型，2个A类型，刚好能够用于Step 1: 融合侧环 (脂肪并环)的构建！假设构建的脂肪并环为：11-24（-23-22）-24-（23-24（-23-22）-23-23）-24-24（-23-22）-11，需要消耗5个24号，3个22号，2个11号和6个23号！明显的，目前的22号不足，无法完成脂肪并环的构建，因此需要调整24号的结构单元分布！

首先考虑转化1个25号为24号（如果25号的数量本身不为0，且目前少于脂肪碳总数的1.5%,则不转化；至少保留1个25号），对应的增加了H的数量，需要进行H的调整（策略放在后面）！每次转化一个25号为24号（注意转化得到的24的类型根据25号的类型决定，如果一个1-hop存在芳香结构的25号转化24号，则这个24号视为A类，如果一个1-hop不存在芳香结构的25号转化24号，则这个24号视为C类），以前面为例，这样就变为1个25和6个24！这样1个25号的连接需要消耗1个25，3个22号，1个11号和2个23号；假设这个转化的这个24号是C类，恰好6个24号中4个C类，2个A类，刚好能够用于脂肪并环的构建！11-24（-23-22）-24-（23-24（-23-22）-24（-23-22）-23）-24-24（-23-22）-11，消耗6个24号，4个22号，2个11号和6个23号，这样就刚好消耗了7个22号，满足分支的分配条件！及25和24均能够实现正确的资源分配！进行后续的资源分配阶段评估！如果这个24号是A类，则该24号无法用于脂肪并环的构建，只能作为Step 4:单个24支链，及这6个24号组成1个包括5个24号的脂肪并环和1条1个24号的脂肪支链，假设该支链为11-24（-23-22）-23-23-22，消耗1个11,2个22,3个23；加上5个24号构成的脂肪并环11-24（-23-22）-24-（23-24（-23-22）-23-23）-24-24（-23-22）-11需要消耗5个24号，3个22号，2个11号和6个23号，总共需要消耗8个22号。这样22号还是不够，但是25号已经不能继续转化！欠缺一个24号（欠缺数量少于4个），因此考虑将一个23号转为22号，这样会增加H，需要进行H的调整（策略放在后面）！这样再次尝试24号的资源分配，如果通过进行后续的资源分配阶段评估！ 如果不通过，则尝试继续转化1个25号为24号，继续评估，直到24号的分配评估通过或者25号少于脂肪碳总数的1%或者只有1个25号了，则停止转化！

对于25号无法继续转为24号的情况下，且当前的24号欠缺22号的数量超过4个（但少于等于8个），需要将24转为22号（优先24号，然后才是14转为的24号），优先级B>D>A>C的24号为22号！这样会导致H的增加，需要进行H的调整（策略放在后面）！例如，假设目前还存在1个25号，9个24号（A类3个，B类1个，C类4个，D类1个），20个23号，5个22号，20个11号。25号的连接形式11-25-23-23-22 + 22 + 22消耗1个25，3个22号，1个11号和2个23号。按照24号分支类型，Step 1: 融合侧环 (脂肪并环)，Step 2: 竖直环 (上下脂肪环)，Step 3: 侧环 (侧边脂肪环)，Step 4: 单个24支链，确定当前9个24号的分支分配为1个脂肪并环（2个A类，2个固定C类，2个C类），1个上下脂肪环（1个固定A类，1个D类），1个单个24支链（1个B类），依次按照4个类型的顺序确定当前的24号能够组合那些分支类型！然后计算需要消耗的资源（支链先取最短-23-22），及脂肪并环11-24（-23-22）-24-（23-24（-23-22）-24（-23-22）-23）-24-24（-23-22）-11消耗6个24号，4个22号，2个11号和6个23号，和上下脂肪环11-24-23-24（-22）-23-23-23-（24，回到第一个24）消耗2个24号，1个22号，1个11号和4个23号，和单个24支链11-24（-22）-23-23-22消耗1个24号，2个22号，1个11号和2个23号，总计需要消耗9个24号，7个22号，4个11号和12个23号。加上25号消耗的资源，总共需要消耗1个25，9个24号，10个22号，5个11号和14个23号。目前只有5个22号，欠缺5个22号（大于4个少于等于8个），且25不能继续转化。只能转化1个24号为22号，优先级B>D>A>C，目前转化1个B类24为22类，增加的H注意调整！这样就只有8个24号了，对24号按照分支类型分配，只能分配1个脂肪并环和1个上下脂肪环，不再有单个24支链！减少2个22号的消耗！算上25号只需要8个22号，当前转化之后的22号存在6个，相差2个，直接将欠缺的2个23号转为22号即可，增加的H注意调整！然后进行24号的分支分配评估，如果通过则进行后续的资源分配阶段评估，如果不通过则继续将1个24号转为22号，然后直到24号的分配评估通过或者只有2个24号了，则停止转化！

如果25号无法继续转为24号的情况下，且当前的24号欠缺22号的数量超过8个，需要将24转为23号（优先24号，然后才是14转为的24号），优先级B>D>A>C的24号转为23号，增加的H注意调整，
然后进行24号的分支分配评估，如果通过则进行后续的资源分配阶段评估，如果不通过则计算差多少的22号，超过8个则继续将1个24号转为23号，如果超过4个但少于等于8个，则将24转为22号，如果少于4个则直接将23转为22，均需要注意H的补正！

这个过程中涉及H的调整，逻辑如下：扩大H的容忍范围为±4%，可以依次循环尝试将13号转为12号结构单元或者23号转为13号结构单元来实现H的减少，或者将一对14/15/16（14/15/16成对转化，如14-15,15-16,15-15，但是注意16的数量不能超过14/15的总和，且14/15/16的总数低于5%的13号时就停止14/15/16的转化，同时14号的数量不超过15号数量）转化为1个12和1个13！例如，目前存在6个14,10个15,4个16号（14/15/16的总数是偶数，因为他们成对出现，也成对用于转化），由于欠缺3个22号，3个23转为了22号导致增加3个H（增加的H数量超出±4%范围），依次将1个13号转为12号，减少一个H，查看是否满足H的容忍范围！满足就不再转化，不满足继续将1个23转为13号，减少1个H继续评估是否满足H的容忍范围，不满足继续尝试将1个13转为12号减少1个H，还是不满足则继续尝试将1个23转为13号来减少一个H！（已经两轮将13转12，23转13，下一轮就是成对的如15-16转为一个12和一个13，减少两个H，或者15-15转为一个12和一个13，减少一个H；然后回到13转12，23转13，每两轮13转12，23转13就进行一轮成对的14/15/16转为12/13，如果14/15/16满足了限制就不在进行轮动，只进行13转12，23转13;注意这样的轮动，不是单次H的调整才进行轮动，而是继承性的，每次需要H的调整，都需要基于前面的轮动情况来决定本次的轮动）。

这个过程中还可能涉及11号和23号的不足情况！如果是11号不足，则单次循环将1个12号转为11号，然后评估是否满足分配，不满足则继续将1个13号转为11号（减少H），然后评估是否满足分配，不满足则循环将1个12号转为11号评估（12转11,13转11分别轮流进行转化），直到满足分支分配。如果是23号不足，每次将2个13号转为1个23号和1个12号，然后评估当前的分支资源分配是否满足，不满足继续将2个13号转为1个23号和1个12号，直到满足分支分配。

注意每次进行了结构单元的调整，需要立刻进行H的补正，然后进行资源分配的重新评估！例如，假设由于24号欠缺了3个22号导致无法完成24号的分支分配评估，由于欠缺的数量少于3个，因此直接23转为22号即可。调整的流程应该是这样的：每次转化都是只转化1个23为22，然后进行H的审查和补正，再进行资源分配的重新评估，发现当前24号的分配评估不通过，继续转化1个1个23为22，然后进行H的审查和补正，再进行资源分配的重新评估，发现当前24号的分配评估还是不通过，继续只转化1个23为22，然后进行H的审查和补正，再进行资源分配的重新评估，发现当前24号的分配评估通过，停止转化。就是说每次的调整实际上就只调整1个结构单元（其中，对于23号不足的情况，是每次将2个13号转为1个23号和1个12），然后立刻进行H的补正和资源分配的重新评估，基于评估结果决定是否继续调整。不是一次性调整到位！需要频繁的调用H的补正和资源分配的评估函数！

#### 柔性/侧链及11/22调整
先前已经完成了24/25号结构单元的调整，实现了分支资源的正确分配！还可能存在11号过多/过少的情况，因此需要进一步调整11号的数量。11过多表现为，当完成分支资源分配之后，进行到Phase 5: 剩余资源分配 和Phase 5b: 过量23重新分配 (_redistribute_excess_23) 处理未使用的11/23/22时，最终发现构建的柔性链11-23-...23-11很多/很少。如何评价柔性链过多/过少：首先计算当前的芳香结构单元能够组合得到多少的芳香团簇！使用转化后的13号和12号组合得到芳香团簇[Aromatic Conversion]Converted 13: 292 (from 5-11,13 + 26 + 2×30)，Converted 12: 90 (rounded up to even)，相应的逻辑在代码RL_init.py中存在，可以调用改代码来实现计算，类似对于资源分配代码的调用！假设计算得到的芳香团簇为X个，然后基于刚性连接的结构单元10号数量计算刚性连接对Y=10号数量/2(取整)！最终的刚性团簇数量为Z=X-Y.注意Z不能为负数和0！（及刚性连接对的数量少于芳香团簇的数量，如果超过，将10号转为12号，直到Y小于X）

那么柔性链的数量应该控制在X*0.8（取整数）以内但必须大于等于Z，如果柔性链的数量超过X*0.8（取整数）或者低于Z+1，那么就需要调整11号的数量。调整逻辑如下：

首先尝试单次转化2个23转为1个17和1个18（减少3个H），同时将3个11转为3个13（增加3个H），满足H的守恒转化，无需进行H的补正！然后重新进行资源分配的评估（完整的资源分配评估，柔性链的数量应该控制在X*0.8（取整数）以内但必须大于等于Z+1，且Z+1必须小于等于X*0.8（取整数），如果Z+1大于X*0.8（取整数），则柔性链的数量等于Z+1）；如果不满足评估，继续尝试单次单次转化2个23转为1个17和1个18（减少3个H），同时将3个11转为3个13（增加3个H），如果17+18的总数超过3%的芳香结构单元总数（5-13号），则不再尝试这类转化，否则继续转化，并重复上述的评估流程，直到满足评估条件或者17+18的总数超过3%的芳香结构单元总数（5-13号）。此外，还需要满足脂肪类结构单元的最低数量要求，及19-25号的脂肪碳总数需要大于等于n（先前基于光谱0-90脂肪区域和90-160芳香区域和160-240羰基区域的比例，面积比，假设为x，y，z，C的总数为N，则n=（4.425+x*0.123+x*x*0.00754）*N），如少于n，则需要停止23号的转化！

如果17+18的总数超过3%的芳香结构单元总数（5-13号）但不满足评估条件，则可以将2个23号转为2个15号（减少2个H），同时将2个11转为1个13（增加2个H），满足H的守恒转化，无需进行H的补正！然后重新进行资源分配的评估（完整的资源分配评估，柔性链的数量应该控制在X*0.8（取整数）以内但必须大于等于Z+1，且Z+1必须小于等于X*0.8（取整数），如果Z+1大于X*0.8（取整数），则柔性链的数量等于Z+1）；如果不满足评估，继续尝试单次单次转化2个23转为2个15号（减少2个H），同时将1个11转为1个13，一个23号转为22号（增加3个H），如果14+15+16的总数超过6%的芳香结构单元总数（5-13号），则不再尝试这类转化，否则继续转化，并重复上述的评估流程，直到满足评估条件或者14+15+16的总数超过6%的芳香结构单元总数（5-13号）。此外，还需要满足脂肪类结构单元的最低数量要求，及19-25号的脂肪碳总数需要大于等于n（先前基于光谱0-90脂肪区域和90-160芳香区域和160-240羰基区域的比例，面积比，假设为x，y，z，C的总数为N，则n=（4.425+x*0.123+x*x*0.00754）*N），如少于n，则需要停止23号的转化！

如果14+15+16的总数超过6%的芳香结构单元总数但还是不满足评估条件，则可以进入一下的轮次循环：单次将1个11和1个23转为2个13号，满足H的守恒转化，无需进行H的补正！然后重新进行资源分配的评估（完整的资源分配评估，柔性链的数量应该控制在X*0.8（取整数）以内但必须大于等于Z+1，且Z+1必须小于等于X*0.8（取整数），如果Z+1大于X*0.8（取整数），则柔性链的数量等于Z+1）；如果不满足评估，继续尝试单次单次转化1个11和1个23转为2个13号，然后重新进行资源分配的评估；如果不满足评估，则单次将2个23号转为1个13号和1个22号，满足H的守恒转化，无需进行H的补正！然后重新进行资源分配的评估。这样先尝试连续两轮进行1个11和1个23转为2个13号，每次评估资源分配不通过，然后进行一轮2个23号转为1个13号和1个22号，再次评估资源分配不通过，这样三个轮次构成轮次循环！下一次转化，回到连续两轮进行1个11和1个23转为2个13号，直到满足资源分配评估条件。此外，还需要满足脂肪类结构单元的最低数量要求，及19-25号的脂肪碳总数需要大于等于n（先前基于光谱0-90脂肪区域和90-160芳香区域和160-240羰基区域的比例，面积比，假设为x，y，z，C的总数为N，则n=（4.425+x*0.123+x*x*0.00754）*N），如少于n，则需要停止23号的转化！

如果发现23号已经触底最低的n了，但是依旧没有满足柔性链的数量应该控制在X*0.8（取整数）以内但必须大于等于Z+1，且Z+1必须小于等于X*0.8（取整数），如果Z+1大于X*0.8（取整数），则柔性链的数量等于Z+1的要求！则尝试将1个11号转为1个13号，增加1个H（前提是H没有超过+4%的最终上限），然后重新进行资源分配的评估，如果不通过，则尝试将1个11号转为1个13号，增加1个H，然后重新进行资源分配的评估，及轮次循环，先1个11号转为1个13号（前提是H没有超过+4%的最终上限），评估不通过就将1个11号转为1个12号，还是不通过则回到1个11号转为1个13号，直到满足柔性链的数量应该控制在X*0.82（取整数）以内以内但必须大于等于Z的要求！

最后如果当前的H没有超过+4%(最终上限)，可以在H容忍范围分别将13和11号转为23号（因为大量的23号被转化了，可能导致23号的不足），转化增加的H最多增加2%的H，例如当前的H是+1%<4%，最多增加2%的H，及上限为3%的H，假设距离+3%还存在5个H的容忍范围，那么依次轮流将1个13号转为23号（+1H），然后1个11号转为23号（+2H），然后继续1个13号转为23号（+1H），1个11号转为23号（+2H）直到达到3%的H容忍范围！同样的，如果当前的H是-2%，那么最多通过转化13和11号转为23号增加2%的H，及H上限是0%。同样的，如果当前的H是3%，那么通过转化13和11号转为23号只允许增加1%的H，因为H最大上限是4%。

完成资源分配评估后，实现结构单元的调整，然后进入循环Layer1-2-3！


In [ ]:
#spectrum_csv = Path("/mnt/e/NN/GA/LY/standardized_nmr.csv")
#elements_str = "C=600,H=350,O=27,N=7,S=4,X=0"

  --spectrum_csv /home/dofengqi/Graphnmr/standardized_nmr.csv \
  --elements "C=460,H=400,O=60,N=2,S=2,X=0" \

  --spectrum_csv /mnt/e/NN/GA/LY/standardized_nmr.csv \
  --elements "C=600,H=350,O=27,N=7,S=4,X=0" \

  --spectrum_csv /mnt/e/NN/GA/XLT/standardized_nmr.csv \
  --elements "C=300,H=355,O=62,N=7,S=8,X=0" \

  --spectrum_csv /mnt/e/NN/GA/MAS/standardized_nmr.csv \
  --elements "C=560,H=328,O=20,N=10,S=4,X=0" \

  --spectrum_csv /mnt/e/NN/GA/NT/standardized_nmr.csv \
  --elements "C=560,H=436,O=46,N=10,S=4,X=0" \


In [ ]:
!python model/z_library.py filter \
  -i z_library/subgraph_library.pt \
  -o z_library/subgraph_library_filtered.pt 

In [ ]:
!python3 test_v1.py \
  --spectrum_csv /home/dofengqi/Graphnmr/standardized_nmr.csv \
  --elements "C=460,H=400,O=60,N=2,S=2,X=0" \
  --output_dir test_results/HSQ-1 \
  --eval_nmr \
  --enable_hop1_adjust \
  --enable_layer3_approx_hop2 \
  --carbonyl_cycles 3 \
  --skeleton_cycles 1 \
  --final_smooth_sigma_ppm 1.2 \
  --final_smooth_passes 3